<a href="https://colab.research.google.com/github/anweshatripathy37-debug/A-python/blob/main/Dmpm_asgn_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download the 'netflix-shows' dataset from Kaggle
path = kagglehub.dataset_download("shivamb/netflix-shows")

# Print the path where the dataset was downloaded
print(f"Dataset downloaded to: {path}")

Using Colab cache for faster access to the 'netflix-shows' dataset.
Dataset downloaded to: /kaggle/input/netflix-shows


## Introduction to the Netflix Shows Dataset

This dataset contains a comprehensive list of movies and TV shows available on Netflix, along with their associated metadata such as cast, director, country, release year, duration, and ratings. We will use `pandas` to load and explore this dataset to understand its structure and content.

In [ ]:
import pandas as pd
import os

# List the contents of the downloaded directory to find the data file
dataset_files = os.listdir(path)
print(f"Files in dataset directory: {dataset_files}")

# Assuming the main data file is a CSV and named 'netflix_titles.csv'
# If the filename is different, this code may need adjustment.
data_file = os.path.join(path, 'netflix_titles.csv')

# Load the dataset into a pandas DataFrame
df = pd.read_csv(data_file)

# Display the first 5 rows of the DataFrame
display(df.head())

Files in dataset directory: ['netflix_titles.csv']


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


### Dataset Overview: Shape, Info, and Missing Values

Before diving into detailed analysis, let's examine the basic characteristics of our dataset. We'll check its dimensions (number of rows and columns), review the data types of each column, and get a summary of non-null values to understand data completeness.

In [ ]:
# Display the shape of the DataFrame (rows, columns)
print(f"Dataset shape: {df.shape}\n")

# Get a concise summary of the DataFrame, including data types and non-null values
print("DataFrame Info:")
df.info()

# Check for missing values in each column
print("\nMissing values per column:")
print(df.isnull().sum())

Dataset shape: (8807, 12)

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB

Missing values per column:
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating       

### Handling Missing Values and Data Type Conversion

Based on the initial overview, several columns contain missing values and some are not in their optimal data types. In this step, we will:

1.  Fill missing values in `director`, `cast`, `country`, and `rating` columns with 'Unknown' to maintain data integrity and prevent errors in later analysis.
2.  Convert the `date_added` column to a proper datetime format, as it is crucial for time-series analysis. Missing `date_added` values will be filled with a placeholder, e.g., the mode or a specific date if necessary for further steps. For simplicity, I'll fill with a common date if there are NaNs after conversion.
3.  Process the `duration` column to separate the numerical value from its unit ('min' or 'Seasons') and convert the numerical part to an integer type.

In [ ]:
# Fill missing values in 'director', 'cast', 'country' and 'rating' with 'Unknown'
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')
df['rating'] = df['rating'].fillna('Unknown')

# Convert 'date_added' to datetime format
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

# Fill missing 'date_added' values (if any remain after coercion) with the most frequent date
if df['date_added'].isnull().any():
    most_frequent_date = df['date_added'].mode()[0]
    df['date_added'] = df['date_added'].fillna(most_frequent_date)

# --- Revised 'duration' processing ---

# Initialize new columns with pandas' dedicated NA (not a number) value
df['duration_min'] = pd.NA
df['num_seasons'] = pd.NA

# Process 'Movie' durations
movie_mask = df['type'] == 'Movie'
# Extract numeric part, coerce errors to NaN, fill NaNs with 0, then convert to int
df.loc[movie_mask, 'duration_min'] = df.loc[movie_mask, 'duration'].str.replace(' min', '', regex=False)
df['duration_min'] = pd.to_numeric(df['duration_min'], errors='coerce').fillna(0).astype(int)

# Process 'TV Show' durations
tvshow_mask = df['type'] == 'TV Show'
# Extract numeric part, coerce errors to NaN, fill NaNs with 0, then convert to int
df.loc[tvshow_mask, 'num_seasons'] = df.loc[tvshow_mask, 'duration'].str.replace(' Season[s]?', '', regex=True)
df['num_seasons'] = pd.to_numeric(df['num_seasons'], errors='coerce').fillna(0).astype(int)

# Drop the original 'duration' column
df.drop('duration', axis=1, inplace=True)

# Verify changes
print("\nDataFrame Info after preprocessing:")
df.info()
print("\nMissing values after preprocessing:")
print(df.isnull().sum())


DataFrame Info after preprocessing:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   show_id       8807 non-null   object        
 1   type          8807 non-null   object        
 2   title         8807 non-null   object        
 3   director      8807 non-null   object        
 4   cast          8807 non-null   object        
 5   country       8807 non-null   object        
 6   date_added    8807 non-null   datetime64[ns]
 7   release_year  8807 non-null   int64         
 8   rating        8807 non-null   object        
 9   listed_in     8807 non-null   object        
 10  description   8807 non-null   object        
 11  duration_min  8807 non-null   int64         
 12  num_seasons   8807 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(9)
memory usage: 894.6+ KB

Missing values after preprocessing

In [ ]:
# Display the first few rows with the new duration columns
display(df.head())

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
